In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ============================================================
# DGCNN + DEVIGN (CLASS-IMBALANCE FIXED)
# Dataset: dgcnn-devign0
# ============================================================

import os, json, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# --------------------
# DEVICE
# --------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# DATASET PATH
# ============================================================
DATASET_PATH = "/kaggle/input/dgcnn-devign0"

print("\nFiles in dataset:")
print(os.listdir(DATASET_PATH))

# ============================================================
# LOAD DEVIGN CSV
# ============================================================
def load_devign_csv(path):
    df = pd.read_csv(path)
    if "func" in df.columns:
        df["code"] = df["func"]
    if "target" in df.columns:
        df["label"] = df["target"]
    df = df[["code", "label"]]
    df["label"] = df["label"].astype(int)
    return df

train_df = load_devign_csv(f"{DATASET_PATH}/devignx_train.csv")
val_df   = load_devign_csv(f"{DATASET_PATH}/Devignx_validation.csv")
test_df  = load_devign_csv(f"{DATASET_PATH}/devignx_test.csv")

print("\nLabel distribution (train):")
print(train_df["label"].value_counts())

# ============================================================
# CLASS WEIGHTS (🔥 KEY FIX)
# ============================================================
counts = train_df["label"].value_counts().sort_index()
class_weights = torch.tensor(
    [counts[1] / counts.sum(), counts[0] / counts.sum()],
    dtype=torch.float
).to(device)

print("\nClass weights:", class_weights)

# ============================================================
# TOKENIZER
# ============================================================
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
MAX_LEN = 256

class DevignDataset(Dataset):
    def __init__(self, df):
        self.codes = df["code"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.codes[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(DevignDataset(train_df), batch_size=32, shuffle=True)
test_loader  = DataLoader(DevignDataset(test_df), batch_size=32)

# ============================================================
# DGCNN MODEL
# ============================================================
class DGCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, k=8):
        super().__init__()
        self.k = k
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(embed_dim, 128, 3, padding=1)
        self.conv2 = nn.Conv1d(128, 128, 3, padding=1)
        self.conv3 = nn.Conv1d(128, 128, 3, padding=1)
        self.fc1 = nn.Linear(128 * k, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 2)

    def kmax_pool(self, x):
        x, _ = torch.topk(x, self.k, dim=2)
        return x

    def forward(self, x):
        x = self.embedding(x).permute(0, 2, 1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.kmax_pool(x).reshape(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

model = DGCNN(tokenizer.vocab_size).to(device)

# ============================================================
# TRAINING SETUP (🔥 LOWER LR)
# ============================================================
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
EPOCHS = 8

# ============================================================
# TRAINING
# ============================================================
print("\nTraining DGCNN on Devign...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)
        loss = criterion(model(ids), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# ============================================================
# EVALUATION
# ============================================================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for batch in test_loader:
        ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)
        preds = torch.argmax(model(ids), dim=1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\nPrediction distribution:", np.unique(y_pred, return_counts=True))

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print("\n===== DGCNN DEVIGN RESULTS =====")
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_true, y_pred, zero_division=0))
print("FPR      :", fp / (fp + tn) if (fp + tn) > 0 else 0)
print("Confusion Matrix:", tn, fp, fn, tp)


Device: cuda
GPU: Tesla T4

Files in dataset:
['Devignx_validation.csv', 'devignx_test.csv', 'devignx_train.csv']

Label distribution (train):
label
0    10356
1     8766
Name: count, dtype: int64

Class weights: tensor([0.4584, 0.5416], device='cuda:0')


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]


Training DGCNN on Devign...
Epoch 1/8 | Loss: 0.6909
Epoch 2/8 | Loss: 0.6847
Epoch 3/8 | Loss: 0.6770
Epoch 4/8 | Loss: 0.6695
Epoch 5/8 | Loss: 0.6615
Epoch 6/8 | Loss: 0.6541
Epoch 7/8 | Loss: 0.6459
Epoch 8/8 | Loss: 0.6372

Prediction distribution: (array([0, 1]), array([ 835, 1897]))

===== DGCNN DEVIGN RESULTS =====
Accuracy : 0.5428257686676428
Precision: 0.5007907221929362
Recall   : 0.7587859424920128
F1 Score : 0.6033661479834869
FPR      : 0.6398648648648648
Confusion Matrix: 533 947 302 950
